# nb2 — Phase 1: Pilot bảng âm tiết (P1) + mô hình nhiễu (P2)

Notebook thứ ba trong chuỗi nb0→nb3 (DESIGN.md §7), gánh cả 2 pilot của Phase 1:

- **Pilot 1 — Bảng âm tiết + non-word rate** (DESIGN.md §4): dựng bảng âm tiết hợp lệ từ nguồn công khai, đo trên VSEC-val **2 chiều**: (a) % lỗi là non-word — trần recall của detector từ điển; (b) % token câu sạch bị bảng đánh dấu oan — precision ceiling (kiểm chứng giả định "precision gần tuyệt đối" DESIGN.md §3). Có bảng sensitivity (generative ~20k) để đo chênh lệch.
- **Pilot 2 — Mô hình nhiễu tổng hợp**: bootstrap bảng confusion thực nghiệm từ `correction_pairs` **train** + rule tạo biến thể (gõ phím kề, hoán âm vùng miền s/x, tr/ch, r/d/gi, ng/n, đổi thanh/dấu), sinh câu nhiễu từ `corrected_text` train, calibration với lỗi thật, QA 50 mẫu soát tay.

**Không sinh tập augmentation đầy đủ** (quyết định 22/09): nb2 chỉ xuất noise model + mẫu QA — nb3 tự sinh augmentation với ratio là hyperparameter.

Tham chiếu: `DESIGN.md` §3 (2 trục), §4 (Pilot 1/2), §5 (ngưỡng quyết định), §9 (leakage) · `PROJECT.md` §3 (schema VSEC).

**Input**: output nb0 — KHÔNG cần output nb1 (không đo gì trên test, quyết định 22/09). Trên Kaggle: *Add Input* dataset tạo từ nb0; local: `./out`.

**Internet**: chỉ cần khi không có file bảng âm tiết upload (download từ URL pin). Không cài package mới — chỉ stdlib.

## Checklist chống leakage của notebook này (DESIGN.md §9)

- Confusion/noise/demo/QA: **chỉ từ `vsec_train.jsonl`** (assert `split == 'train'` mọi record); val chỉ dùng để **đo** (Pilot 1), không fit.
- Bảng âm tiết: nguồn công khai độc lập, ghi URL/SHA256 vào output; không suy từ val/test.
- Không đọc bất kỳ file test nào; không đọc output nb1.
- Seed cố định 42; mọi tham số gom cell cấu hình; mọi số bị loại/bỏ qua được log.

## Nội dung
0. Cấu hình + tự dò input nb0
1. Cell hàm dùng chung (copy nguyên vẹn từ nb1, `SHARED_CELLS_VERSION`) + helper riêng
2. Pilot 1 — dựng bảng âm tiết (upload > URL pin; sensitivity tùy chọn)
3. Pilot 1 — đo 2 chiều trên VSEC-val (+ sensitivity)
4. Pilot 2 — confusion thực nghiệm từ correction_pairs train
5. Pilot 2 — rule sinh biến thể + noise sampler (seeded)
6. Pilot 2 — sinh demo + calibration với lỗi thật
7. QA 50 mẫu in ra soát tay
8. Xuất 4 file + pilot_report.json
9. Đưa output sang nb3

## 0. Cấu hình — mọi tham số gom một chỗ (DESIGN.md §7)

| Tham số | Giá trị | Ý nghĩa |
|---|---|---|
| `SEED` | `42` | seed cố định cho sampler/QA (tái lập được) |
| `QA_N` | `50` | số mẫu QA in ra soát tay (DESIGN.md §4: 50) |
| `DEMO_N` | `200` | số câu nhiễu demo (calibration + pool QA) |
| `MAX_ERRORS_PER_SENT` | `4` | trần số lỗi/câu khi sinh (theo bin error_count 1/2/3/≥4) |
| `P_EMPIRICAL` | `0.5` | xác suất dùng confusion thực nghiệm khi âm tiết có sẵn |
| `SUSPECT_EDIT_RATIO` | `0.3` | giữ nguyên giá trị nb1 — cell hàm dùng chung copy nguyên vẹn cần nó |
| `DICT_PRIMARY_URL` | URL pin | 7.184 âm tiết thường gặp (bao >94% lần xuất hiện), repo `vietnameselanguage/syllable` pin theo commit `52ff591` |
| `DICT_SENSITIVITY_URL` | URL pin | ~20k âm tiết generative (onset×rime, gist `hieuthi`) — chỉ để đo sensitivity, download fail thì bỏ qua |
| `DICT_MIN_LINES` | `5000` | ngưỡng nhận file bảng âm tiết upload (chống nhầm file thường) |
| `DICT_VALID_RATIO` | `0.9` | ≥90% dòng phải là 1 token chữ (không số, không khoảng trắng) |

**Input tự dò** (như nb1): quét `/kaggle/input/**/manifest.json` rồi `./out/manifest.json` — cần đủ `manifest.json` + `vsec_train.jsonl` + `vsec_val.jsonl` (KHÔNG cần `test_normalized.jsonl`; thiếu → dừng sớm với hướng dẫn). **File bảng âm tiết upload được ưu tiên hơn URL**: notebook dò `*.txt` trong `/kaggle/input` (ngoài dataset nb0) thỏa `DICT_MIN_LINES` + `DICT_VALID_RATIO` — nếu có thì không cần Internet.

In [16]:
import datetime
import json
from pathlib import Path

SEED = 42
QA_N = 50
DEMO_N = 200
MAX_ERRORS_PER_SENT = 4
P_EMPIRICAL = 0.5
SUSPECT_EDIT_RATIO = 0.3

DICT_PRIMARY_URL = 'https://raw.githubusercontent.com/vietnameselanguage/syllable/52ff591eaec7bea0647cd75f7be66c7acae82c25/vietnamesesyllable_7184_sorted_abc.txt'
DICT_SENSITIVITY_URL = 'https://gist.githubusercontent.com/hieuthi/0f5adb7d3f79e7fb67e0e499004bf558/raw/7af2772a05af89821efd282ce3f4beef49b1a00d/all-vietnamese-syllables.txt'
DICT_MIN_LINES = 5000
DICT_VALID_RATIO = 0.9

REQUIRED_FILES = ['manifest.json', 'vsec_train.jsonl', 'vsec_val.jsonl']


def find_nb0_output():
    candidates = []
    kaggle = Path('/kaggle/input')
    if kaggle.is_dir():
        candidates += sorted(kaggle.rglob('manifest.json'))
    local = Path('./out/manifest.json')
    if local.is_file():
        candidates.append(local)
    for m in candidates:
        if all((m.parent / f).is_file() for f in REQUIRED_FILES):
            return m.parent
    return None


INPUT_DIR = find_nb0_output()
if INPUT_DIR is None:
    raise FileNotFoundError(
        'Không tìm thấy output của nb0 (cần đủ: ' + ', '.join(REQUIRED_FILES) + '). '
        'Trên Kaggle: Add Input dataset đã tạo từ nb0 (xem nb0 §9). Local: chạy nb0 với OUTPUT_DIR ./out.'
    )

OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path('./out')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_STAMP = datetime.datetime.now().isoformat(timespec='seconds')
manifest_nb0 = json.loads((INPUT_DIR / 'manifest.json').read_text(encoding='utf-8'))

print('INPUT_DIR :', INPUT_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('nb0 manifest:', manifest_nb0.get('notebook'), '— tạo', manifest_nb0.get('created'))
print(f'SEED={SEED}  QA_N={QA_N}  DEMO_N={DEMO_N}  P_EMPIRICAL={P_EMPIRICAL}  MAX_ERRORS_PER_SENT={MAX_ERRORS_PER_SENT}')
print('Số câu theo manifest nb0 — train:', manifest_nb0.get('vsec', {}).get('train'),
      '· val:', manifest_nb0.get('vsec', {}).get('val'))

INPUT_DIR : /kaggle/input/notebooks/cquangnguynl/nb0-data-pre
OUTPUT_DIR: /kaggle/working
nb0 manifest: nb0_data_prep — tạo 2026-09-22T07:16:53
SEED=42  QA_N=50  DEMO_N=200  P_EMPIRICAL=0.5  MAX_ERRORS_PER_SENT=4
Số câu theo manifest nb0 — train: 8343 · val: 927


## 1. Cell hàm dùng chung — copy NGUYÊN VẸT từ nb1 (DESIGN.md §7)

Cell dưới là bản copy y hệt cell shared của nb1 (`SHARED_CELLS_VERSION = 'align-v1'`) — không sửa một ký tự nào. nb2 dùng `nfc_normalize`, `canon_tokenize`, `is_punct_token`; các hàm align còn lại không gọi nhưng giữ nguyên để các notebook không lệch nhau (nguyên tắc DESIGN.md §7: bị sửa ở ≥2 nơi thì trích `.py` chung).

Helper riêng của nb2 đặt ở cell kế (không thuộc phần copy): `load_jsonl`, `is_word_token` (bỏ dấu câu + token chứa số khi đo/sinh lỗi), `is_nonword` (tra bảng âm tiết, NFC + lower), `sha256_text`, `download_text` (stdlib, không pip install).

In [17]:
import re
import unicodedata

SHARED_CELLS_VERSION = 'align-v1'

_WS_RE = re.compile(r'\s+')
_TOKEN_RE = re.compile(r'\w+|[^\w\s]+')
_HAS_WORD_RE = re.compile(r'\w')


def nfc_normalize(s):
    return _WS_RE.sub(' ', unicodedata.normalize('NFC', s)).strip()


def canon_tokenize(s):
    """NFC + tách token: run chữ/số liên liền (\\w+) là 1 token; cụm dấu câu liền kề là 1 token riêng.
    Chuẩn hóa "style VSEC" (dấu câu tách space) triệt để — dùng THỐNG NHẤT cho VSEC / test / output
    model (DESIGN.md §2.4). So sánh token là case-sensitive ("tronG" ≠ "trong")."""
    return _TOKEN_RE.findall(nfc_normalize(s))


def is_punct_token(tok):
    return not _HAS_WORD_RE.search(tok)


def levenshtein_opcodes(src, tgt):
    """Levenshtein DP thuần Python (sub/ins/del = 1) trên 2 chuỗi token.
    Trả về opcodes kiểu difflib: list (tag, i1, i2, j1, j2) với tag ∈ {equal, replace, insert, delete},
    phủ kín src[i1:i2] ↔ tgt[j1:j2], không chồng lấn, theo thứ tự xuất hiện."""
    n, m = len(src), len(tgt)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        dp[i][0] = i
    for j in range(1, m + 1):
        dp[0][j] = j
    for i in range(1, n + 1):
        si = src[i - 1]
        prev, row = dp[i - 1], dp[i]
        for j in range(1, m + 1):
            best = prev[j - 1] + (0 if si == tgt[j - 1] else 1)  # match / substitute
            if prev[j] + 1 < best:  # delete token nguồn
                best = prev[j] + 1
            if row[j - 1] + 1 < best:  # insert token đích
                best = row[j - 1] + 1
            row[j] = best
    ops = []

    def _push(tag, i1, i2, j1, j2):
        if ops and ops[-1][0] == tag and ops[-1][1] == i2 and ops[-1][3] == j2:
            ops[-1] = (tag, i1, ops[-1][2], j1, ops[-1][4])
        else:
            ops.append((tag, i1, i2, j1, j2))

    i, j = n, m
    while i > 0 or j > 0:
        if i > 0 and j > 0 and dp[i][j] == dp[i - 1][j - 1] + (0 if src[i - 1] == tgt[j - 1] else 1):
            _push('equal' if src[i - 1] == tgt[j - 1] else 'replace', i - 1, i, j - 1, j)
            i, j = i - 1, j - 1
        elif i > 0 and dp[i][j] == dp[i - 1][j] + 1:
            _push('delete', i - 1, i, j, j)
            i -= 1
        else:
            _push('insert', i, i, j - 1, j)
            j -= 1
    ops.reverse()
    return ops


def _make_block(src, tgt, span):
    i1, i2, j1, j2 = span
    src_toks, tgt_toks = src[i1:i2], tgt[j1:j2]
    ns, nt = len(src_toks), len(tgt_toks)
    if ns == 0:
        btype = 'insert'
    elif nt == 0:
        btype = 'delete'
    elif ns == 1 and nt == 1:
        btype = 'substitute'
    elif ns == 1:
        btype = 'split'
    elif nt == 1:
        btype = 'merge'
    else:
        btype = 'multi'
    return {
        'type': btype,
        'position': i1,
        'src_span': [i1, i2],
        'tgt_span': [j1, j2],
        'src_tokens': src_toks,
        'tgt_tokens': tgt_toks,
        'punct_only': all(is_punct_token(t) for t in src_toks + tgt_toks),
    }


def extract_edit_blocks(src, tgt, opcodes):
    """Gộp các opcode không-equal liền kề thành 1 edit block (chịu được split/merge/dịch vị trí).
    type theo số token (nguồn → đích): substitute 1-1 · split 1→n · merge m→1 · insert 0→n ·
    delete m→0 · multi m→n. position = index token nguồn đầu tiên của block
    (insert: vị trí chèn trước, có thể == len(src))."""
    blocks, cur = [], None
    for tag, i1, i2, j1, j2 in opcodes:
        if tag == 'equal':
            if cur is not None:
                blocks.append(_make_block(src, tgt, cur))
                cur = None
        elif cur is None:
            cur = [i1, i2, j1, j2]
        else:
            cur[1], cur[3] = i2, j2
    if cur is not None:
        blocks.append(_make_block(src, tgt, cur))
    return blocks


def build_pseudo_annotation(text, corrected, suspect_ratio=SUSPECT_EDIT_RATIO):
    """Align text ↔ corrected_text trong hệ token canonical → pseudo-annotation schema giống VSEC + edit_blocks.
    Quy ước QUAN TRỌNG (nb3 phải dùng cell này nguyên vẹn):
    - error_positions: index token nguồn nằm trong src_span của block CÓ token nguồn.
      Block insert (thiếu âm tiết ở nguồn) KHÔNG có vị trí nguồn → không nằm trong error_positions,
      chỉ nằm trong correction_pairs với error='' và position = vị trí chèn trước (có thể == len(src)).
    - correction_pairs: 1 entry/block; error/correction = các token nối bằng space; delete → correction=''.
    - syllable_annotations: 1 entry/token nguồn; is_correct=False khi token thuộc src_span của block
      non-insert; corrections = chuỗi đích (join space) của block đó.
    - suspect: edit_ratio = (tổng token cả 2 vế nằm trong edit block) / max(len(src), len(tgt)) vượt ngưỡng.
    - align_failed: một trong hai vế token hóa rỗng."""
    src = canon_tokenize(text)
    tgt = canon_tokenize(corrected)
    blocks = extract_edit_blocks(src, tgt, levenshtein_opcodes(src, tgt))
    corrections_by_pos = {}
    for b in blocks:
        if b['src_span'][0] < b['src_span'][1]:
            fix = ' '.join(b['tgt_tokens'])
            for i in range(b['src_span'][0], b['src_span'][1]):
                corrections_by_pos.setdefault(i, []).append(fix)
    error_positions = sorted(corrections_by_pos)
    syllable_annotations = [
        {
            'syllable': tok,
            'is_correct': i not in corrections_by_pos,
            'corrections': corrections_by_pos.get(i, []),
            'position': i,
        }
        for i, tok in enumerate(src)
    ]
    correction_pairs = [
        {'error': ' '.join(b['src_tokens']), 'correction': ' '.join(b['tgt_tokens']), 'position': b['position']}
        for b in blocks
    ]
    n_edit_tokens = sum(
        (b['src_span'][1] - b['src_span'][0]) + (b['tgt_span'][1] - b['tgt_span'][0]) for b in blocks
    )
    denom = max(len(src), len(tgt))
    edit_ratio = n_edit_tokens / denom if denom else 0.0
    return {
        'is_clean': not blocks,
        'align_failed': not src or not tgt,
        'suspect': edit_ratio > suspect_ratio,
        'edit_ratio': round(edit_ratio, 4),
        'error_count': len(blocks),
        'error_positions': error_positions,
        'correction_pairs': correction_pairs,
        'syllable_annotations': syllable_annotations,
        'edit_blocks': blocks,
        'src_tokens': src,
        'tgt_tokens': tgt,
    }


def apply_edit_blocks(src, blocks):
    """Roundtrip invariant: áp edit blocks vào src tokens → phải thu về đúng tgt tokens."""
    out, pos = [], 0
    for b in blocks:
        out += src[pos:b['src_span'][0]]
        out += b['tgt_tokens']
        pos = b['src_span'][1]
    out += src[pos:]
    return out


print('SHARED_CELLS_VERSION:', SHARED_CELLS_VERSION)

SHARED_CELLS_VERSION: align-v1


In [18]:
import collections
import hashlib
import random
import urllib.request

_DICT_LINE_RE = re.compile(r'[^\W\d_]+')
_DIGIT_RE = re.compile(r'\d')


def load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]


def is_word_token(tok):
    """Token tính vào đo lường/sinh lỗi: không phải dấu câu và không chứa chữ số."""
    return not is_punct_token(tok) and not _DIGIT_RE.search(tok)


def is_nonword(tok, syll_set):
    return nfc_normalize(tok).lower() not in syll_set


def sha256_text(text):
    return hashlib.sha256(text.encode('utf-8')).hexdigest()


def download_text(url, timeout=30):
    req = urllib.request.Request(url, headers={'User-Agent': 'nb2-pilot/1.0'})
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        return resp.read().decode('utf-8')


print('Helpers nb2 sẵn sàng')

Helpers nb2 sẵn sàng


## 2. Pilot 1 — dựng bảng âm tiết

Ưu tiên nguồn (quyết định 22/09): **file upload > URL pin**:

1. Tự dò file `*.txt` trong `/kaggle/input` không thuộc dataset nb0, có ≥`DICT_MIN_LINES` dòng không rỗng và ≥`DICT_VALID_RATIO` trong số đó là 1 token chữ (không số, không khoảng trắng) → dùng làm bảng chính (không cần Internet).
2. Không có → download `DICT_PRIMARY_URL` (pin bằng SHA256 nội dung ghi vào output).

Chuẩn hóa: NFC → lower → bỏ entry rỗng/chứa số/không phải 1 token chữ → dedupe → sort. Sanity **fail cứng** (chặn đo pilot bằng bảng sai): kích thước 5.000–25.000; spot-check 10 âm tiết tiếng Việt phổ biến — thiếu >1/10 → dừng (chống nhầm wordlist script khác qua đường upload); in phân bố độ dài.

Bảng **sensitivity** (generative ~20k, onset×rime — tính chất khác: chứa cả âm tiết hợp âm vị học nhưng không bao giờ xuất hiện thật) download riêng: fail thì bỏ gracefully, notebook vẫn chạy trọn vẹn.

Xuất `syllable_table.json` (entries + metadata nguồn) ngay tại section này.

In [19]:
def looks_like_dict_file(path):
    lines = [ln.strip() for ln in path.read_text(encoding='utf-8').splitlines() if ln.strip()]
    if len(lines) < DICT_MIN_LINES:
        return False
    ok = sum(1 for ln in lines if _DICT_LINE_RE.fullmatch(ln))
    return ok / len(lines) >= DICT_VALID_RATIO


def find_uploaded_dict():
    kaggle = Path('/kaggle/input')
    if not kaggle.is_dir():
        return None
    for p in sorted(kaggle.rglob('*.txt')):
        if (p.parent / 'vsec_train.jsonl').is_file():
            continue
        try:
            if looks_like_dict_file(p):
                return p
        except (UnicodeDecodeError, OSError):
            continue
    return None


def build_table(raw_text):
    entries = set()
    for ln in raw_text.splitlines():
        s = nfc_normalize(ln).lower()
        if s and _DICT_LINE_RE.fullmatch(s):
            entries.add(s)
    return sorted(entries)


dict_sources = []
uploaded_dict = find_uploaded_dict()
if uploaded_dict is not None:
    dict_raw = uploaded_dict.read_text(encoding='utf-8')
    dict_sources.append({'kind': 'upload', 'path': str(uploaded_dict)})
    print('Dùng bảng âm tiết upload:', uploaded_dict)
else:
    try:
        dict_raw = download_text(DICT_PRIMARY_URL)
        dict_sources.append({'kind': 'url', 'url': DICT_PRIMARY_URL})
        print('Đã tải bảng âm tiết từ:', DICT_PRIMARY_URL)
    except Exception as exc:
        raise RuntimeError(
            f'Không tải được bảng âm tiết ({DICT_PRIMARY_URL}): {exc}. '
            'Cần Internet ON, hoặc upload file danh sách âm tiết (>=5000 dòng, mỗi dòng 1 âm tiết chữ) '
            'lên Kaggle Dataset rồi Add Input vào notebook này.'
        ) from exc

SYLLABLES = build_table(dict_raw)
SYLL_SET = set(SYLLABLES)
dict_sources[-1].update({
    'sha256': sha256_text(dict_raw),
    'n_raw': len([ln for ln in dict_raw.splitlines() if ln.strip()]),
    'n_kept': len(SYLLABLES),
})

print(f'Bảng chính: {dict_sources[-1]["n_raw"]} dòng thô → {len(SYLLABLES)} âm tiết (NFC, lower, dedupe)')
if not (DICT_MIN_LINES <= len(SYLLABLES) <= 25000):
    raise RuntimeError(f'Kích thước bảng âm tiết bất thường: {len(SYLLABLES)} (kỳ vọng {DICT_MIN_LINES}-25000) — dừng thay vì đo pilot bằng bảng sai')

missing = [w for w in ['người', 'việt', 'xanh', 'học', 'nghiêng', 'quyết', 'trường', 'chính', 'viên', 'tả'] if w not in SYLL_SET]
print('Spot-check 10 âm tiết phổ biến:', 'đủ cả' if not missing else f'THIẾU {missing}')
if len(missing) > 1:
    raise RuntimeError(f'Bảng âm tiết thiếu {len(missing)}/10 âm tiết tiếng Việt phổ biến {missing} — nghi là bảng không phải tiếng Việt, dừng thay vì đo pilot bằng bảng sai')
print('Phân bố độ dài âm tiết:', dict(sorted(collections.Counter(len(s) for s in SYLLABLES).items())))

SENSITIVITY_OK = False
SYLLABLES_GEN = []
try:
    raw_gen = download_text(DICT_SENSITIVITY_URL)
    SYLLABLES_GEN = build_table(raw_gen)
    SENSITIVITY_OK = True
    print(f'Bảng sensitivity (generative): {len(SYLLABLES_GEN)} âm tiết — SHA256 {sha256_text(raw_gen)[:12]}')
except Exception as exc:
    print(f'[Thông báo] Bỏ qua bảng sensitivity: {exc} — nb vẫn chạy trọn vẹn với bảng chính.')

with open(OUTPUT_DIR / 'syllable_table.json', 'w', encoding='utf-8') as f:
    json.dump({'created': RUN_STAMP, 'sources': dict_sources,
               'sensitivity': {'ok': SENSITIVITY_OK, 'n': len(SYLLABLES_GEN)},
               'n': len(SYLLABLES), 'entries': SYLLABLES}, f, ensure_ascii=False)
print('Đã ghi syllable_table.json vào', OUTPUT_DIR)

Đã tải bảng âm tiết từ: https://raw.githubusercontent.com/vietnameselanguage/syllable/52ff591eaec7bea0647cd75f7be66c7acae82c25/vietnamesesyllable_7184_sorted_abc.txt
Bảng chính: 7884 dòng thô → 7884 âm tiết (NFC, lower, dedupe)
Spot-check 10 âm tiết phổ biến: đủ cả
Phân bố độ dài âm tiết: {1: 74, 2: 1028, 3: 3172, 4: 2560, 5: 887, 6: 157, 7: 6}
Bảng sensitivity (generative): 17974 âm tiết — SHA256 78eeb840d504
Đã ghi syllable_table.json vào /kaggle/working


## 3. Pilot 1 — đo 2 chiều trên VSEC-val (không đụng test — quyết định 22/09)

- **(a) Error side — trần recall của detector từ điển**: với mỗi `correction_pairs` của val, tokenize vế `error`, bỏ token dấu câu: đúng 1 token chữ → non-word nếu ∉ bảng; 1 token chứa số → skip riêng `skipped_digit` (đồng bộ với clean-side dùng `is_word_token`); ≥2 token → nhóm **structural** (split/merge/insert/delete theo PROJECT.md §3.2 — báo cáo riêng); rỗng → skip (log). Non-word rate = nonword / (nonword + realword). In top-20 lỗi non-word (ứng viên bắt bằng tra bảng cho hướng C) và top-20 lỗi real-word (cần ngữ cảnh — neural lo).
- **(b) Clean side — precision ceiling**: mọi token chữ của `corrected_text` val → % ∉ bảng. In top-20 token bị flag (kỳ vọng: tên riêng, từ mượn, viết tắt) — đối chiếu giả định "precision gần tuyệt đối" (DESIGN.md §3).
- **Sensitivity**: chạy lại (a)+(b) với bảng generative → in chênh lệch (nếu tải được).
- Kết luận thô theo ngưỡng DESIGN.md §5 (non-word ≥ ~40% → C có nền; thấp → bỏ C sớm). Notebook chỉ **báo số** — quyết định cuối thuộc người dùng.

In [20]:
train_records = load_jsonl(INPUT_DIR / 'vsec_train.jsonl')
val_records = load_jsonl(INPUT_DIR / 'vsec_val.jsonl')
print(f'train: {len(train_records)} câu · val: {len(val_records)} câu')


def measure_error_side(records, syll_set):
    res = collections.Counter()
    nonword_top = collections.Counter()
    realword_top = collections.Counter()
    for rec in records:
        for pair in rec.get('correction_pairs') or []:
            toks = [t for t in canon_tokenize(pair.get('error', '')) if not is_punct_token(t)]
            if not toks:
                res['skipped_empty'] += 1
            elif len(toks) == 1 and not is_word_token(toks[0]):
                res['skipped_digit'] += 1
            elif len(toks) == 1:
                if is_nonword(toks[0], syll_set):
                    res['nonword'] += 1
                    nonword_top[toks[0].lower()] += 1
                else:
                    res['realword'] += 1
                    realword_top[toks[0].lower()] += 1
            else:
                res['structural'] += 1
    res['pairs_total'] = sum(res[k] for k in ['skipped_empty', 'skipped_digit', 'nonword', 'realword', 'structural'])
    res['nonword_top'] = nonword_top.most_common(20)
    res['realword_top'] = realword_top.most_common(20)
    return res


def measure_clean_side(records, syll_set):
    res = collections.Counter()
    oov_top = collections.Counter()
    for rec in records:
        for tok in canon_tokenize(rec['corrected_text']):
            if not is_word_token(tok):
                continue
            res['tokens_total'] += 1
            if is_nonword(tok, syll_set):
                res['oov'] += 1
                oov_top[tok] += 1
    res['oov_top'] = oov_top.most_common(20)
    return res


def nonword_rate(m):
    sub = m['nonword'] + m['realword']
    return m['nonword'] / sub if sub else 0.0


val_err = measure_error_side(val_records, SYLL_SET)
val_clean = measure_clean_side(val_records, SYLL_SET)
rate = nonword_rate(val_err)
oov_rate = val_clean['oov'] / val_clean['tokens_total'] if val_clean['tokens_total'] else 0.0

print('== Pilot 1 — VSEC-val ==')
print(f"(a) Error side : {val_err['nonword']} non-word / {val_err['realword']} real-word / "
      f"{val_err['structural']} structural / {val_err['skipped_empty']} skip / {val_err['skipped_digit']} skip-digit "
      f"→ non-word rate = {rate:.1%}")
print('    Top lỗi non-word :', [t for t, _ in val_err['nonword_top'][:15]])
print('    Top lỗi real-word:', [t for t, _ in val_err['realword_top'][:15]])
print(f"(b) Clean side : {val_clean['oov']}/{val_clean['tokens_total']} token bị flag ({oov_rate:.2%}) "
      '— precision ceiling của detector từ điển')
print('    Top token bị flag:', [t for t, _ in val_clean['oov_top'][:15]])
print()
print('Kết luận thô theo ngưỡng DESIGN.md §5 (>=40% → C có nền; thấp → bỏ C sớm):')
print('  → non-word rate', f'{rate:.1%}', '— hybrid từ điển (C) CÓ NỀN' if rate >= 0.4 else '— cân nhắc bỏ C sớm')

sensitivity_stats = None
if SENSITIVITY_OK:
    gen_set = set(SYLLABLES_GEN)
    gen_err = measure_error_side(val_records, gen_set)
    gen_clean = measure_clean_side(val_records, gen_set)
    sensitivity_stats = {
        'n_table': len(SYLLABLES_GEN),
        'nonword_rate': nonword_rate(gen_err),
        'clean_oov_rate': gen_clean['oov'] / gen_clean['tokens_total'] if gen_clean['tokens_total'] else 0.0,
    }
    print()
    print('== Sensitivity (bảng generative) ==')
    print(f"non-word rate {sensitivity_stats['nonword_rate']:.1%} vs {rate:.1%} (bảng chính) · "
          f"clean OOV {sensitivity_stats['clean_oov_rate']:.2%} vs {oov_rate:.2%}")

train: 8343 câu · val: 927 câu
== Pilot 1 — VSEC-val ==
(a) Error side : 252 non-word / 858 real-word / 1 structural / 0 skip / 1 skip-digit → non-word rate = 22.7%
    Top lỗi non-word : ['gía', 'nôị', 'qúa', 'tâp', 'dộ', 'cac', 'só', 'hoc', 'vâỵ', 'vầ', 'cở', 'nguời', 'triễn', 'đạng', 'duc']
    Top lỗi real-word: ['đô', 'các', 'khoẻ', 'hang', 'thoả', 'trong', 'dung', 'qua', 'uỷ', 'trinh', 'tính', 'nhưng', 'thế', 'họp', 'thể']
(b) Clean side : 456/27708 token bị flag (1.65%) — precision ceiling của detector từ điển
    Top token bị flag: ['HS', 'TH', 'KH', 'Honda', 'GV', 'Internet', 'True', 'Milk', 'FDI', 'DN', 'online', 'ODA', 'T', 'C', 'BĐKH']

Kết luận thô theo ngưỡng DESIGN.md §5 (>=40% → C có nền; thấp → bỏ C sớm):
  → non-word rate 22.7% — cân nhắc bỏ C sớm

== Sensitivity (bảng generative) ==
non-word rate 15.4% vs 22.7% (bảng chính) · clean OOV 2.36% vs 1.65%


## 4. Pilot 2 — confusion thực nghiệm từ `correction_pairs` train

**Assert mọi record `split == 'train'`** (dòng phòng vệ leakage — fail sớm nếu nhầm file). Giữ pair khi cả `error` và `correction` tokenize ra đúng 1 token chữ mỗi vế → đếm `confusion[correction][error]` (hướng đúng để áp lên câu sạch: correction → error). Pair multi-token (split/merge/insert/delete, PROJECT.md §3.2) bị loại có thống kê + mẫu in ra.

Tổng hợp: số âm tiết sạch có ≥1 nhầm lẫn đã biết, top-30 pairs (nhìn pattern vùng miền s/x, tr/ch, r/d/gi, ng/n...), error-vocabulary (tập mọi âm tiết từng xuất hiện ở vế lỗi — dùng làm bộ lọc candidate ở §5).

In [21]:
for rec in train_records:
    assert rec.get('split') == 'train', f'Leakage: record row_id={rec.get("row_id")} không phải split=train'
print(f'Assert PASS: toàn bộ {len(train_records)} record là split=train')

confusion = {}
confusion_stats = {'pairs_total': 0, 'kept': 0, 'dropped_empty': 0, 'dropped_multi_token': 0, 'dropped_punct_digit': 0}
drop_samples = []
for rec in train_records:
    for pair in rec.get('correction_pairs') or []:
        confusion_stats['pairs_total'] += 1
        e_toks = canon_tokenize(pair.get('error', ''))
        c_toks = canon_tokenize(pair.get('correction', ''))
        if not e_toks or not c_toks:
            confusion_stats['dropped_empty'] += 1
        elif len(e_toks) != 1 or len(c_toks) != 1:
            confusion_stats['dropped_multi_token'] += 1
            if len(drop_samples) < 5:
                drop_samples.append((pair.get('error'), pair.get('correction')))
        elif not (is_word_token(e_toks[0]) and is_word_token(c_toks[0])):
            confusion_stats['dropped_punct_digit'] += 1
        else:
            low_c = c_toks[0].lower()
            low_e = e_toks[0].lower()
            confusion.setdefault(low_c, {})
            confusion[low_c][low_e] = confusion[low_c].get(low_e, 0) + 1
            confusion_stats['kept'] += 1

ERROR_VOCAB = {e for errs in confusion.values() for e in errs}
top_pairs = sorted(((c, e, n) for c, errs in confusion.items() for e, n in errs.items()), key=lambda x: -x[2])[:30]

print('== Pilot 2 — confusion thực nghiệm (train) ==')
print(confusion_stats)
print(f'Số âm tiết sạch có >=1 nhầm lẫn đã biết: {len(confusion)} · error-vocabulary: {len(ERROR_VOCAB)} âm tiết')
print('Top-30 pairs (sửa → lỗi, số lần):')
print('  ' + ' · '.join(f'{c}→{e}({n})' for c, e, n in top_pairs))
print('Mẫu pair multi-token bị loại:', drop_samples)

Assert PASS: toàn bộ 8343 record là split=train
== Pilot 2 — confusion thực nghiệm (train) ==
{'pairs_total': 10039, 'kept': 8915, 'dropped_empty': 314, 'dropped_multi_token': 805, 'dropped_punct_digit': 5}
Số âm tiết sạch có >=1 nhầm lẫn đã biết: 1285 · error-vocabulary: 2843 âm tiết
Top-30 pairs (sửa → lỗi, số lần):
  thỏa→thoả(55) · dùng→dung(49) · những→nhưng(48) · hệ→hê(47) · hiện→hiên(45) · ủy→uỷ(44) · hàng→hang(39) · trọng→trong(39) · đề→để(38) · cách→các(37) · thể→thế(37) · thống→thông(37) · tòa→toà(34) · định→đinh(33) · được→đươc(33) · động→đông(33) · khỏe→khoẻ(32) · tăng→tang(31) · cơ→cở(30) · để→đề(29) · bộ→bô(29) · hòa→hoà(28) · nỗ→nổ(28) · khóa→khoá(27) · nên→lên(27) · của→cuả(26) · dụng→dung(26) · thế→thể(25) · được→dược(25) · hưởng→hướng(25)
Mẫu pair multi-token bị loại: [('nửa.', 'nữa.'), ('nghĩaa,', 'nghĩa'), ('trính,', 'trình'), ('đuwowcj.', 'được.'), ('(bản', 'bán')]


## 5. Pilot 2 — rule sinh biến thể + noise sampler

**Rule** (chỉ substitute 1-1, quyết định 22/09; mỗi rule = hàm sinh candidate từ 1 âm tiết thường):

| Rule | Cơ chế | Ví dụ |
|---|---|---|
| `regional` | hoán âm đầu theo cặp phương ngữ: s↔x, tr↔ch, r↔d↔gi, n↔l, ng↔n, ngh↔ng (chỉ áp khi ký tự sau là nguyên âm) | `xanh→sanh`, `trong→chong`, `răng→dăng` |
| `tone` | hoán thanh điệu giữa 5 thanh có dấu (huyền/sắc/hỏi/ngã/nặng) qua NFD/NFC | `đẹp→đẻp` |
| `vowel` | hoán dấu phụ nguyên âm: a↔ă↔â, o↔ô↔ơ, u↔ư, e↔ê | `học→hôc` |
| `keyboard` | thay/bỏ/đúp 1 ký tự ASCII theo bảng kề QWERTY (edit distance 1) | `trong→tron`, `anh→anhh` |

**Bộ lọc candidate** (chấp nhận nếu 1 trong): ∈ bảng âm tiết (real-word) ∨ ∈ error-vocabulary train (non-word người thật từng gõ) ∨ edit-distance ký tự = 1. Mục tiêu: không sinh rác vô lý ("nghười").

**Sampler** (seeded, deterministic): số lỗi/câu k ~ phân bố `error_count` train (cap `MAX_ERRORS_PER_SENT`); chọn k vị trí trong token chữ có candidate; mỗi vị trí: confusion thực nghiệm với xác suất `P_EMPIRICAL` (nếu âm tiết có sẵn), else rule; ép candidate ≠ gốc; giữ hoa/thường của token gốc. Invariant: chạy 2 lần cùng seed → output identical; câu sinh khác gốc **đúng k** vị trí đã chọn.

In [22]:
VOWEL_CHARS = set('aăâeêioôơuưy')
VOWEL_MAP = {'a': 'ăâ', 'ă': 'aâ', 'â': 'aă', 'e': 'ê', 'ê': 'e', 'o': 'ôơ', 'ô': 'oơ', 'ơ': 'oô', 'u': 'ư', 'ư': 'u'}
TONE_MARKS = [chr(0x300), chr(0x301), chr(0x303), chr(0x309), chr(0x323)]

QWERTY_ADJ = {
    'q': 'wa', 'w': 'qeas', 'e': 'wrsd', 'r': 'etdf', 't': 'ryfg', 'y': 'tugh',
    'u': 'yihj', 'i': 'uojk', 'o': 'ipkl', 'p': 'ol',
    'a': 'qwsz', 's': 'awdxz', 'd': 'sefcx', 'f': 'drgvc', 'g': 'fthbv', 'h': 'gyujn',
    'j': 'hikmn', 'k': 'jilm', 'l': 'kop',
    'z': 'asx', 'x': 'zsdc', 'c': 'xdfv', 'v': 'cfgb', 'b': 'vghn', 'n': 'bhjm', 'm': 'njk',
}

REGIONAL_SWAPS = [
    ('s', 'x'), ('x', 's'), ('tr', 'ch'), ('ch', 'tr'),
    ('r', 'd'), ('d', 'r'), ('d', 'gi'), ('gi', 'd'), ('r', 'gi'), ('gi', 'r'),
    ('n', 'l'), ('l', 'n'), ('ng', 'n'), ('n', 'ng'), ('ngh', 'ng'), ('ng', 'ngh'),
]


def regional_variants(s):
    out = []
    for src, dst in REGIONAL_SWAPS:
        if s.startswith(src):
            rest = s[len(src):]
            if rest and rest[0] in VOWEL_CHARS:
                out.append(dst + rest)
    return out


def tone_variants(s):
    d = unicodedata.normalize('NFD', s)
    for mark in TONE_MARKS:
        if mark in d:
            return [unicodedata.normalize('NFC', d.replace(mark, m)) for m in TONE_MARKS if m != mark]
    return []


def vowel_variants(s):
    out = []
    for i, ch in enumerate(s):
        for alt in VOWEL_MAP.get(ch, ''):
            if alt != ch:
                out.append(s[:i] + alt + s[i + 1:])
    return out


def keyboard_variants(s):
    out = []
    for i, ch in enumerate(s):
        if ch.isascii() and ch.isalpha():
            for rep in QWERTY_ADJ.get(ch.lower(), ''):
                out.append(s[:i] + rep + s[i + 1:])
            out.append(s[:i] + s[i + 1:])
            out.append(s[:i] + ch + s[i:])
    return out


RULE_FUNCS = [
    ('regional', regional_variants),
    ('tone', tone_variants),
    ('vowel', vowel_variants),
    ('keyboard', keyboard_variants),
]


def rule_candidates(syl):
    cands = {}
    for name, fn in RULE_FUNCS:
        for c in fn(syl):
            if c != syl and c not in cands:
                cands[c] = name
    return cands


def char_lev_at_most_1(a, b):
    if abs(len(a) - len(b)) > 1:
        return False
    if len(a) < len(b):
        a, b = b, a
    i = j = diff = 0
    while i < len(a) and j < len(b):
        if a[i] == b[j]:
            i += 1
            j += 1
        else:
            diff += 1
            if diff > 1:
                return False
            if len(a) == len(b):
                i += 1
                j += 1
            else:
                i += 1
    if i < len(a) or j < len(b):
        diff += 1
    return diff <= 1


print('Rule sẵn sàng:', [name for name, _ in RULE_FUNCS])
_demo_cands = rule_candidates('xanh')
assert 'sanh' in _demo_cands and _demo_cands['sanh'] == 'regional', _demo_cands
assert char_lev_at_most_1('tron', 'trong') and char_lev_at_most_1('anhh', 'anh')
assert not char_lev_at_most_1('abc', 'a')
print('Sanity rule PASS — rule_candidates("xanh"):', _demo_cands)

Rule sẵn sàng: ['regional', 'tone', 'vowel', 'keyboard']
Sanity rule PASS — rule_candidates("xanh"): {'sanh': 'regional', 'xănh': 'vowel', 'xânh': 'vowel', 'zanh': 'keyboard', 'danh': 'keyboard', 'canh': 'keyboard', 'anh': 'keyboard', 'xxanh': 'keyboard', 'xqnh': 'keyboard', 'xwnh': 'keyboard', 'xsnh': 'keyboard', 'xznh': 'keyboard', 'xnh': 'keyboard', 'xaanh': 'keyboard', 'xabh': 'keyboard', 'xahh': 'keyboard', 'xajh': 'keyboard', 'xamh': 'keyboard', 'xah': 'keyboard', 'xannh': 'keyboard', 'xang': 'keyboard', 'xany': 'keyboard', 'xanu': 'keyboard', 'xanj': 'keyboard', 'xann': 'keyboard', 'xan': 'keyboard', 'xanhh': 'keyboard'}


In [23]:
ERROR_COUNT_DIST = collections.Counter(min(rec.get('error_count') or 1, MAX_ERRORS_PER_SENT) for rec in train_records)


def accepted_rule_candidates(low_tok, syll_set):
    cands = {}
    for c, src in rule_candidates(low_tok).items():
        if c != low_tok and (c in syll_set or c in ERROR_VOCAB or char_lev_at_most_1(c, low_tok)):
            cands[c] = 'rule:' + src
    return cands


def generate_noisy(clean_text, rng, syll_set):
    toks = canon_tokenize(clean_text)
    cand_cache = {}
    eligible = []
    for i, tok in enumerate(toks):
        if not is_word_token(tok):
            continue
        low = nfc_normalize(tok).lower()
        opts = {}
        for e in confusion.get(low, {}):
            if e != low:
                opts[e] = 'empirical'
        for c, src in accepted_rule_candidates(low, syll_set).items():
            opts.setdefault(c, src)
        if opts:
            eligible.append(i)
            cand_cache[i] = (low, opts)
    if not eligible:
        return None
    k = min(rng.choices(list(ERROR_COUNT_DIST), weights=list(ERROR_COUNT_DIST.values()), k=1)[0], len(eligible))
    out = list(toks)
    edits = []
    for i in sorted(rng.sample(eligible, k)):
        low, opts = cand_cache[i]
        emp = {c: confusion[low][c] for c, s in opts.items() if s == 'empirical'}
        if emp and rng.random() < P_EMPIRICAL:
            pick = rng.choices(list(emp), weights=list(emp.values()), k=1)[0]
        else:
            rules = [c for c, s in opts.items() if s.startswith('rule:')]
            if rules:
                pick = rng.choice(rules)
            else:
                pick = rng.choices(list(emp), weights=list(emp.values()), k=1)[0]
        cand = pick[:1].upper() + pick[1:] if toks[i][:1].isupper() else pick
        out[i] = cand
        edits.append({'position': i, 'original': toks[i], 'error': cand, 'source': opts[pick]})
    assert len(edits) == k
    return {'clean_text': ' '.join(toks), 'noisy_text': ' '.join(out), 'edits': edits}


_det_a = [generate_noisy(train_records[i]['corrected_text'], random.Random(SEED + 1), SYLL_SET) for i in range(10)]
_det_b = [generate_noisy(train_records[i]['corrected_text'], random.Random(SEED + 1), SYLL_SET) for i in range(10)]
assert _det_a == _det_b, 'Sampler không deterministic với cùng seed!'
print('Sampler deterministic PASS (10 câu, cùng seed → output identical)')
print('Phân bố số lỗi/câu train (cap 4):', dict(sorted(ERROR_COUNT_DIST.items())))

Sampler deterministic PASS (10 câu, cùng seed → output identical)
Phân bố số lỗi/câu train (cap 4): {1: 7016, 2: 1089, 3: 172, 4: 66}


## 6. Pilot 2 — sinh demo + calibration với lỗi thật

Sinh `DEMO_N` câu nhiễu từ tập con seeded của `corrected_text` train (câu không tạo được vị trí sinh nào bị skip, có log). Calibration:

1. **Non-word rate**: lỗi sinh (tra bảng Pilot 1) vs lỗi thật trên **train** (đo cùng phép §3a nhưng trên train — cùng tập so sánh).
2. **Số lỗi/câu**: phân bố sinh vs phân bố thật (lưu ý: sinh bị giới hạn bởi số vị trí có candidate).
3. **Nguồn lỗi**: tỷ lệ empirical vs rule (và theo từng rule).

Lệch mạnh → ghi nhận trong report; hiệu chỉnh `P_EMPIRICAL`/trọng số rule là việc Phase 2, không chặn nb2. Kiểm chứng invariant trên toàn bộ demo: câu sinh khác gốc **đúng tại** các vị trí đã chọn.

In [24]:
rng = random.Random(SEED)
demo_pool = []
demo_skipped = 0
for text in rng.sample([r['corrected_text'] for r in train_records], min(DEMO_N * 2, len(train_records))):
    if len(demo_pool) >= DEMO_N:
        break
    g = generate_noisy(text, rng, SYLL_SET)
    if g is None:
        demo_skipped += 1
        continue
    demo_pool.append(g)

for g in demo_pool:
    changed = [i for i, (a, b) in enumerate(zip(g['clean_text'].split(), g['noisy_text'].split())) if a != b]
    assert changed == [e['position'] for e in g['edits']], g['clean_text'][:60]
print(f'Đã sinh {len(demo_pool)} câu nhiễu demo (skip {demo_skipped} câu không có vị trí sinh được) — invariant vị trí PASS')

train_err = measure_error_side(train_records, SYLL_SET)
real_rate = nonword_rate(train_err)
syn_edits = [e for g in demo_pool for e in g['edits']]
syn_nonword = sum(1 for e in syn_edits if is_nonword(e['error'], SYLL_SET))
syn_rate = syn_nonword / len(syn_edits) if syn_edits else 0.0
src_dist = collections.Counter(e['source'] for e in syn_edits)
k_syn = collections.Counter(len(g['edits']) for g in demo_pool)
k_real = collections.Counter(min(r.get('error_count') or 1, MAX_ERRORS_PER_SENT) for r in train_records)

print('== Calibration ==')
print(f'Non-word rate: lỗi sinh {syn_rate:.1%} vs lỗi thật train {real_rate:.1%}')
print('Số lỗi/câu (sinh):', dict(sorted(k_syn.items())))
print('Số lỗi/câu (thật):', dict(sorted(k_real.items())))
print('Nguồn lỗi        :', dict(src_dist.most_common()))
print(f'Lệch non-word rate {abs(syn_rate - real_rate):.1%} — lệch mạnh thì ghi nhận, hiệu chỉnh ở Phase 2')

Đã sinh 200 câu nhiễu demo (skip 0 câu không có vị trí sinh được) — invariant vị trí PASS
== Calibration ==
Non-word rate: lỗi sinh 51.6% vs lỗi thật train 23.4%
Số lỗi/câu (sinh): {1: 158, 2: 37, 3: 4, 4: 1}
Số lỗi/câu (thật): {1: 7016, 2: 1089, 3: 172, 4: 66}
Nguồn lỗi        : {'rule:keyboard': 114, 'empirical': 113, 'rule:tone': 16, 'rule:vowel': 3, 'rule:regional': 2}
Lệch non-word rate 28.2% — lệch mạnh thì ghi nhận, hiệu chỉnh ở Phase 2


## 7. QA 50 mẫu — in ra soát tay

Chọn seeded từ demo pool (span lỗi đánh dấu `[..]`). Khi soát từng mẫu cần kiểm (rubric):

1. Lỗi sinh có **giống lỗi người thật** (gõ nhầm/phương ngữ) không?
2. **Vị trí** lỗi có hợp lý (rải đều, không dồn vào 1 kiểu từ) không?
3. Câu nhiễu **còn đọc hiểu được** (không vỡ cấu trúc) không?
4. **Đa dạng** loại lỗi (non-word/real-word, keyboard/regional/tone/vowel) có giống VSEC không?

Tiêu chí quyết định DESIGN.md §5: nhiễu khó phân biệt với lỗi thật khi soát tay → trục dữ liệu (pha sạch + nhiễu) khả thi. Danh sách mẫu ghi vào `noise_qa_samples.json` để đối chiếu lại sau.

In [25]:
qa_rng = random.Random(SEED)
qa_pick = sorted(qa_rng.sample(range(len(demo_pool)), min(QA_N, len(demo_pool))))
qa_samples = []
print('Soát tay từng mẫu theo rubric §7 (giống lỗi thật? vị trí? đọc được? đa dạng?):')
print()
for idx in qa_pick:
    g = demo_pool[idx]
    bad_pos = {e['position'] for e in g['edits']}
    marked = ' '.join('[' + t + ']' if i in bad_pos else t for i, t in enumerate(g['noisy_text'].split()))
    print(f'—— QA demo_idx={idx}')
    print('  SẠCH :', g['clean_text'])
    print('  NHIỄU:', marked)
    for e in g['edits']:
        print(f"    @{e['position']}: {e['original']} → {e['error']} ({e['source']})")
    print()
    qa_samples.append({'demo_index': idx, 'clean_text': g['clean_text'], 'noisy_text': g['noisy_text'], 'edits': g['edits']})

print(f'Đã in {len(qa_samples)} mẫu QA (mục tiêu {QA_N}, seed {SEED}).')

Soát tay từng mẫu theo rubric §7 (giống lỗi thật? vị trí? đọc được? đa dạng?):

—— QA demo_idx=1
  SẠCH : Đồng thời , để khảo sát lại các mô hình và triển khai các dự án nông thôn mới , tại mỗi làng đều có văn phòng đại diện Saemaul cùng với các Phòng Nông nghiệp và Phát triển Nông thôn phối hợp làm việc chặt chẽ .
  NHIỄU: Đồng thời , để khảo sát [lạk] các mô hình và triển khai các dự án nông thôn mới , tại mỗi làng đều có văn phòng đại diện Saemaul cùng với các Phòng Nông nghiệp và Phát triển Nông thôn phối hợp làm việc chặt chẽ .
    @6: lại → lạk (rule:keyboard)

—— QA demo_idx=6
  SẠCH : Có hai phương pháp để đo lường đa cộng tuyến như sau : + Tính độ chấp nhận của biến ( Tolerance ): Độ chấp nhận của biến càng nhỏ thì dấu hiệu có đa cộng tuyến càng sâu .
  NHIỄU: Có hai phương pháp để đo [pường] đa cộng tuyến như sau : + Tính độ chấp nhận của biến ( Tolerance ): Độ chấp nhận của biến càng nhỏ thì dấu hiệu có đa cộng tuyến càng sâu .
    @6: lường → pường (rule:keyboard)

—— QA de

## 8. Xuất file + pilot_report.json

- `syllable_table.json` — đã ghi ở §2 (entries + nguồn + SHA256).
- `noise_model.json` — confusion counts + cấu hình rule (bảng QWERTY/vowel/regional/tone) + `P_EMPIRICAL` + seed + `shared_cells_version` + SHA256 bảng âm tiết. nb3 load file này + bảng âm tiết rồi **tự sinh augmentation** theo ratio (hyperparameter) — tái lập được nhờ seed.
- `noise_qa_samples.json` — các mẫu QA đầy đủ metadata.
- `pilot_report.json` — config, nguồn dict + SHA256, Pilot 1 (2 chiều + sensitivity + top lists), Pilot 2 (confusion + calibration), tham chiếu manifest nb0.

In [26]:
noise_model = {
    'created': RUN_STAMP,
    'notebook': 'nb2_pilot_dict_noise',
    'shared_cells_version': SHARED_CELLS_VERSION,
    'seed': SEED,
    'config': {
        'p_empirical': P_EMPIRICAL,
        'max_errors_per_sent': MAX_ERRORS_PER_SENT,
        'rules': [name for name, _ in RULE_FUNCS],
        'qwerty_adj': QWERTY_ADJ,
        'vowel_map': VOWEL_MAP,
        'regional_swaps': REGIONAL_SWAPS,
        'tone_marks': [hex(ord(m)) for m in TONE_MARKS],
    },
    'dict_sha256': dict_sources[-1]['sha256'],
    'error_count_dist': dict(sorted(ERROR_COUNT_DIST.items())),
    'confusion': confusion,
}

with open(OUTPUT_DIR / 'noise_model.json', 'w', encoding='utf-8') as f:
    json.dump(noise_model, f, ensure_ascii=False)

with open(OUTPUT_DIR / 'noise_qa_samples.json', 'w', encoding='utf-8') as f:
    json.dump({'seed': SEED, 'qa_n': QA_N, 'shared_cells_version': SHARED_CELLS_VERSION, 'samples': qa_samples},
              f, ensure_ascii=False, indent=2)

report = {
    'created': RUN_STAMP,
    'notebook': 'nb2_pilot_dict_noise',
    'shared_cells_version': SHARED_CELLS_VERSION,
    'config': {
        'seed': SEED, 'qa_n': QA_N, 'demo_n': DEMO_N,
        'p_empirical': P_EMPIRICAL, 'max_errors_per_sent': MAX_ERRORS_PER_SENT,
        'dict_source_used': dict_sources[0],
        'sensitivity_url': DICT_SENSITIVITY_URL,
    },
    'nb0_manifest_ref': {
        'notebook': manifest_nb0.get('notebook'),
        'created': manifest_nb0.get('created'),
        'vsec_after_dedupe': manifest_nb0.get('vsec', {}).get('after_dedupe'),
        'train': manifest_nb0.get('vsec', {}).get('train'),
        'val': manifest_nb0.get('vsec', {}).get('val'),
    },
    'dict': {
        'sources': dict_sources,
        'n': len(SYLLABLES),
        'sensitivity': {'ok': SENSITIVITY_OK, 'n': len(SYLLABLES_GEN) if SENSITIVITY_OK else None},
    },
    'pilot1': {
        'val': {
            'error_side': {k: val_err[k] for k in ['pairs_total', 'skipped_empty', 'skipped_digit', 'nonword', 'realword', 'structural']},
            'nonword_rate': rate,
            'nonword_top': val_err['nonword_top'],
            'realword_top': val_err['realword_top'],
            'clean_side': {
                'tokens_total': val_clean['tokens_total'],
                'oov': val_clean['oov'],
                'oov_rate': oov_rate,
                'oov_top': val_clean['oov_top'],
            },
        },
        'sensitivity': sensitivity_stats,
        'threshold_40pct': rate >= 0.4,
    },
    'pilot2': {
        'confusion': {
            **confusion_stats,
            'n_clean_syllables': len(confusion),
            'n_error_vocab': len(ERROR_VOCAB),
            'top_pairs': [{'correction': c, 'error': e, 'count': n} for c, e, n in top_pairs],
        },
        'calibration': {
            'demo_n': len(demo_pool),
            'demo_skipped_no_candidate': demo_skipped,
            'synthetic_nonword_rate': syn_rate,
            'train_real_nonword_rate': real_rate,
            'errors_per_sent_synthetic': dict(sorted(k_syn.items())),
            'errors_per_sent_train': dict(sorted(k_real.items())),
            'source_breakdown': dict(src_dist.most_common()),
        },
        'qa': {'n_printed': len(qa_samples), 'target': QA_N},
    },
    'files': ['syllable_table.json', 'noise_model.json', 'noise_qa_samples.json', 'pilot_report.json'],
}
with open(OUTPUT_DIR / 'pilot_report.json', 'w', encoding='utf-8') as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print('Đã ghi vào', OUTPUT_DIR)
print('Files:', report['files'])
print()
print(json.dumps(report, ensure_ascii=False, indent=2))

Đã ghi vào /kaggle/working
Files: ['syllable_table.json', 'noise_model.json', 'noise_qa_samples.json', 'pilot_report.json']

{
  "created": "2026-09-22T09:17:14",
  "notebook": "nb2_pilot_dict_noise",
  "shared_cells_version": "align-v1",
  "config": {
    "seed": 42,
    "qa_n": 50,
    "demo_n": 200,
    "p_empirical": 0.5,
    "max_errors_per_sent": 4,
    "dict_source_used": {
      "kind": "url",
      "url": "https://raw.githubusercontent.com/vietnameselanguage/syllable/52ff591eaec7bea0647cd75f7be66c7acae82c25/vietnamesesyllable_7184_sorted_abc.txt",
      "sha256": "bd55ccfef5ae0c9e85996561aa0a03556b686eaa362a8829e9e2d57c555b3dca",
      "n_raw": 7884,
      "n_kept": 7884
    },
    "sensitivity_url": "https://gist.githubusercontent.com/hieuthi/0f5adb7d3f79e7fb67e0e499004bf558/raw/7af2772a05af89821efd282ce3f4beef49b1a00d/all-vietnamese-syllables.txt"
  },
  "nb0_manifest_ref": {
    "notebook": "nb0_data_prep",
    "created": "2026-09-22T07:16:53",
    "vsec_after_dedupe": 9270

## 9. Đưa output sang nb3

1. Sau khi chạy xong trên Kaggle (*Save Version → Run All*): panel **Output** → chọn 4 file (`syllable_table.json`, `noise_model.json`, `noise_qa_samples.json`, `pilot_report.json`) → **New Dataset** (ví dụ `vsec-pilot2`).
2. nb3: **Add Input** dataset này + dataset output nb0.
3. nb3 copy cell hàm dùng chung nguyên vẹn (`SHARED_CELLS_VERSION` phải khớp `'align-v1'`), load `noise_model.json` + `syllable_table.json`, sinh augmentation **chỉ trên train** với ratio sạch/nhiễu là hyperparameter (sweep ở Phase 2 nếu kịp — factorial 2×2, DESIGN.md §3).
4. Sau khi soát tay 50 mẫu QA + đọc `pilot_report.json` → điền kết quả vào DESIGN.md §11 (mục treo 2+3) và chốt theo ngưỡng §5 — việc của người dùng, không phải notebook.

Bước tiếp theo (DESIGN.md §4): nb3 — baseline LoRA BARTpho fp16, eval 3 chỉ số trên val + test.